# Chest X-Ray Multi-Label Classifier — GPU Training Notebook

**Full NIH ChestX-ray14 pipeline** — config → dataset → model → train → evaluate

### Setup instructions (first time only)
1. Open the Command Palette in VS Code → **Python: Create Environment** → **Venv**
2. Pick a Python 3.12 interpreter when prompted
3. Run the **Environment Setup** cell below — it installs all dependencies
4. Select the new `.venv` kernel in the top-right kernel picker

> **GPU note:** Make sure your CUDA drivers are installed before running. The notebook auto-detects CUDA.

---
## 0 · Environment Setup
Run once to install all required packages into the venv.

In [ ]:
import subprocess, sys

# Core packages — pinned to latest stable (Python 3.12 compatible)
packages = [
    # PyTorch with CUDA 12.1 support — remove +cu121 suffix if CPU-only
    "torch==2.7.1+cu121",
    "torchvision==0.22.1+cu121",
    # The +cu121 index is needed for the GPU wheels
]

other_packages = [
    "scikit-learn>=1.4",
    "pandas>=2.2",
    "Pillow>=10.3",
    "tqdm>=4.66",
    "numpy>=1.26",
    "ipywidgets>=8.1",   # tqdm notebook progress bars
]

# Install PyTorch from the dedicated CUDA index
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
] + packages)

# Install the rest from PyPI
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
] + other_packages)

print("\n✅ All packages installed successfully.")

---
## 1 · Config
**Update `ZIP_PATH`** below to point to your `chest_xray14.zip`. Everything else is auto-resolved.

In [2]:
"""
config
------
Central configuration. Change ZIP_PATH and hyperparameters here.
All other cells read from these variables.
"""

import os
import zipfile

# ─────────────────────────────────────────────
# YOUR ZIP FILE LOCATION  ← update this
# ─────────────────────────────────────────────
ZIP_PATH    = r"C:\Users\adhit\Downloads\neurosymbolic\archive (1).zip"
EXTRACT_DIR = os.path.join(os.path.dirname(ZIP_PATH), "NIH_ChestXray14_extracted")


# ─────────────────────────────────────────────
# AUTO-EXTRACT
# ─────────────────────────────────────────────
def extract_dataset():
    if os.path.isdir(EXTRACT_DIR):
        print(f"[config] Dataset already extracted at:\n         {EXTRACT_DIR}")
        return
    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(
            f"[config] ZIP not found at: {ZIP_PATH}\n"
            "→ Update ZIP_PATH in this cell."
        )
    print("[config] Extracting zip — this may take a minute …")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"[config] Done. Extracted to: {EXTRACT_DIR}")


extract_dataset()


# ─────────────────────────────────────────────
# SMART PATH RESOLUTION
# ─────────────────────────────────────────────
def _find_file(filename):
    for root, dirs, files in os.walk(EXTRACT_DIR):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(
        f"[config] '{filename}' not found inside {EXTRACT_DIR}\n"
        "→ Check that your zip extracted correctly."
    )

_CSV_FOUND     = _find_file("Data_Entry_2017.csv")
DATASET_ROOT   = os.path.dirname(_CSV_FOUND)

DATA_ENTRY_CSV = _CSV_FOUND
TRAIN_VAL_LIST = os.path.join(DATASET_ROOT, "train_val_list_NIH.txt")
TEST_LIST      = os.path.join(DATASET_ROOT, "test_list_NIH.txt")
IMAGES_DIR     = os.path.join(DATASET_ROOT, "images-224", "images-224")
BASE_DIR       = os.getcwd()

for _path, _label in [
    (DATA_ENTRY_CSV,  "Data_Entry_2017.csv"),
    (TRAIN_VAL_LIST,  "train_val_list_NIH.txt"),
    (TEST_LIST,       "test_list_NIH.txt"),
    (IMAGES_DIR,      "images-224/"),
]:
    if not os.path.exists(_path):
        raise FileNotFoundError(f"[config] Missing: {_label}  →  {_path}")

print(f"[config] ✅ All dataset files confirmed at: {DATASET_ROOT}")


# ─────────────────────────────────────────────
# 14 NIH LABELS  (fixed order — do not reorder)
# ─────────────────────────────────────────────
LABELS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Effusion", "Emphysema", "Fibrosis", "Hernia",
    "Infiltration", "Mass", "No Finding", "Pleural Thickening",
    "Pneumonia", "Pneumothorax",
]
NUM_LABELS = len(LABELS)   # 14


# ─────────────────────────────────────────────
# IMAGE SETTINGS
# ─────────────────────────────────────────────
IMG_SIZE = 224
MEAN     = [0.485, 0.456, 0.406]   # ImageNet mean
STD      = [0.229, 0.224, 0.225]   # ImageNet std


# ─────────────────────────────────────────────
# TRAINING HYPERPARAMETERS
# ─────────────────────────────────────────────
BATCH_SIZE   = 96        # increase from 16 → 32 now that GPU has headroom
NUM_EPOCHS   = 10        # full run; bump higher if time allows
LR           = 1e-4
WEIGHT_DECAY = 1e-4


# ─────────────────────────────────────────────
# MODEL & LOSS
# ─────────────────────────────────────────────
MODEL_NAME     = "densenet121"    # densenet121 | resnet50 | resnet18 | vit_b_16 | vit_b_32
USE_PRETRAINED = True
LOSS_TYPE      = "bce_weighted"   # "bce_weighted" or "focal"


# ─────────────────────────────────────────────
# FULL DATASET  (no subset)
# ─────────────────────────────────────────────
SUBSET_SIZE = None   # None = all 112 k images


# ─────────────────────────────────────────────
# CHECKPOINTS
# ─────────────────────────────────────────────
CHECKPOINT_DIR  = os.path.join(BASE_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pth")

print(f"[config] Model       : {MODEL_NAME}")
print(f"[config] Loss        : {LOSS_TYPE}")
print(f"[config] Batch size  : {BATCH_SIZE}")
print(f"[config] Epochs      : {NUM_EPOCHS}")
print(f"[config] Subset size : {SUBSET_SIZE} (None = full dataset)")
print(f"[config] Checkpoints : {CHECKPOINT_DIR}")

[config] Dataset already extracted at:
         C:\Users\adhit\Downloads\neurosymbolic\NIH_ChestXray14_extracted
[config] ✅ All dataset files confirmed at: C:\Users\adhit\Downloads\neurosymbolic\NIH_ChestXray14_extracted
[config] Model       : densenet121
[config] Loss        : bce_weighted
[config] Batch size  : 96
[config] Epochs      : 10
[config] Subset size : None (None = full dataset)
[config] Checkpoints : c:\Users\adhit\Downloads\neurosymbolic\checkpoints


---
## 2 · Dataset

In [3]:
"""
dataset
-------
PyTorch Dataset for NIH ChestX-ray14.
Uses the official patient-level train/val/test split files.
"""

import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T


# ── helpers ──────────────────────────────────
def _load_name_list(filepath):
    """Read a txt file (one filename per line) → set of strings."""
    with open(filepath, "r") as f:
        return {line.strip() for line in f if line.strip()}


def _parse_labels(label_str):
    """
    'Effusion|Pneumonia' → [0,0,0,0,1,0,0,0,0,0,0,0,1,0]
    'No Finding'         → [0,0,0,0,0,0,0,0,0,0,1,0,0,0]
    """
    active = set(label_str.split("|"))
    return [1.0 if lbl in active else 0.0 for lbl in LABELS]


# ── transforms ───────────────────────────────
def get_train_transform():
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD),
        T.RandomErasing(p=0.1, scale=(0.02, 0.08)),
    ])


def get_val_transform():
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD),
    ])


# ── dataset class ─────────────────────────────
class ChestXRayDataset(Dataset):
    """
    Parameters
    ----------
    split       : "train" | "val" | "test"
    subset_size : int or None.  None = full split.
    """

    def __init__(self, split="train", subset_size=SUBSET_SIZE):
        assert split in ("train", "val", "test")

        df = pd.read_csv(DATA_ENTRY_CSV)
        print(f"[dataset] Loaded Data_Entry_2017.csv  →  {len(df):,} rows")

        if split in ("train", "val"):
            allowed = _load_name_list(TRAIN_VAL_LIST)
            df = df[df["Image Index"].isin(allowed)]
            cut = int(len(df) * 0.8)
            df  = df.iloc[:cut] if split == "train" else df.iloc[cut:]
        else:
            allowed = _load_name_list(TEST_LIST)
            df = df[df["Image Index"].isin(allowed)]

        print(f"[dataset] split='{split}'  →  {len(df):,} images (before subset)")

        if subset_size is not None and subset_size < len(df):
            df = df.head(subset_size)
            print(f"[dataset] Subset applied  →  using {len(df):,} images")

        self.image_paths = [os.path.join(IMAGES_DIR, name) for name in df["Image Index"]]
        self.labels      = [_parse_labels(s) for s in df["Finding Labels"]]
        self.transform   = get_train_transform() if split == "train" else get_val_transform()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img   = Image.open(self.image_paths[idx]).convert("RGB")
        img   = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label


# ── quick sanity check ────────────────────────
print("\n--- Dataset sanity check ---")
for _split in ("train", "val", "test"):
    _ds = ChestXRayDataset(split=_split, subset_size=5)
    _img, _lbl = _ds[0]
    print(f"  [{_split}] len={len(_ds)}  image={_img.shape}  label={_lbl.shape}")
print("✅ Dataset OK")


--- Dataset sanity check ---
[dataset] Loaded Data_Entry_2017.csv  →  112,120 rows
[dataset] split='train'  →  69,219 images (before subset)
[dataset] Subset applied  →  using 5 images
  [train] len=5  image=torch.Size([3, 224, 224])  label=torch.Size([14])
[dataset] Loaded Data_Entry_2017.csv  →  112,120 rows
[dataset] split='val'  →  17,305 images (before subset)
[dataset] Subset applied  →  using 5 images
  [val] len=5  image=torch.Size([3, 224, 224])  label=torch.Size([14])
[dataset] Loaded Data_Entry_2017.csv  →  112,120 rows
[dataset] split='test'  →  25,596 images (before subset)
[dataset] Subset applied  →  using 5 images
  [test] len=5  image=torch.Size([3, 224, 224])  label=torch.Size([14])
✅ Dataset OK


---
## 3 · Model & Loss

In [4]:
"""
model
-----
DenseNet-121 (default) with a 14-node linear head.
Also defines FocalLoss and inverse-frequency class-weight helper.
"""

import torch
import torch.nn as nn
import torchvision.models as models


def build_model(model_name=MODEL_NAME, pretrained=USE_PRETRAINED):
    """
    Loads a pretrained backbone and replaces the classifier head
    with a NUM_LABELS-node linear layer.
    Sigmoid is NOT applied here — handled by the loss / inference step.
    """
    weights_arg = lambda w: w if pretrained else None

    if model_name == "densenet121":
        m = models.densenet121(weights=weights_arg("IMAGENET1K_V1"))
        m.classifier = nn.Linear(m.classifier.in_features, NUM_LABELS)

    elif model_name == "resnet50":
        m = models.resnet50(weights=weights_arg("IMAGENET1K_V2"))
        m.fc = nn.Linear(m.fc.in_features, NUM_LABELS)

    elif model_name == "resnet18":
        m = models.resnet18(weights=weights_arg("IMAGENET1K_V1"))
        m.fc = nn.Linear(m.fc.in_features, NUM_LABELS)

    elif model_name == "vit_b_16":
        m = models.vit_b_16(weights=weights_arg("IMAGENET1K_V1"))
        in_f = m.heads.head.in_features
        m.heads = nn.Sequential(nn.LayerNorm(in_f), nn.Linear(in_f, NUM_LABELS))

    elif model_name == "vit_b_32":
        m = models.vit_b_32(weights=weights_arg("IMAGENET1K_V1"))
        in_f = m.heads.head.in_features
        m.heads = nn.Sequential(nn.LayerNorm(in_f), nn.Linear(in_f, NUM_LABELS))

    else:
        raise ValueError(f"Unknown model: '{model_name}'")

    total     = sum(p.numel() for p in m.parameters()) / 1e6
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6
    print(f"[model] '{model_name}'  |  pretrained={pretrained}  |  output={NUM_LABELS} nodes")
    print(f"[model] Params: {total:.1f}M total  |  {trainable:.1f}M trainable")
    return m


class FocalLoss(nn.Module):
    """Multi-label Focal Loss: FL(p) = -α(1-p)^γ log(p)"""

    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )
        focal_w = (1 - torch.exp(-bce)) ** self.gamma
        if self.alpha is not None:
            focal_w = focal_w * self.alpha.to(logits.device)
        return (focal_w * bce).mean()


def compute_class_weights(labels_list):
    """
    Inverse-frequency weights.  w_i = N / (2 * count_i), clipped to [0.1, 50].
    """
    t = torch.tensor(labels_list)
    N = t.shape[0]
    pos = t.sum(dim=0) + 1e-6
    w   = (N / (2.0 * pos)).clamp(0.1, 50.0)
    print(f"[model] Class weights  min={w.min():.2f}  max={w.max():.2f}")
    return w


# ── smoke test ────────────────────────────────
_m = build_model()
_m.eval()
_x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    _out = _m(_x)
print(f"[model] Smoke test output shape: {_out.shape}  ✅")
del _m, _x, _out

[model] 'densenet121'  |  pretrained=True  |  output=14 nodes
[model] Params: 7.0M total  |  7.0M trainable
[model] Smoke test output shape: torch.Size([2, 14])  ✅


---
## 4 · Training
Trains for `NUM_EPOCHS` epochs and saves the best checkpoint (highest val macro-AUC).

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import numpy as np
import os

# ── device ────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[train] Device : {device}")
if device.type == "cuda":
    print(f"[train] GPU    : {torch.cuda.get_device_name(0)}")
    print(f"[train] VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("CUDA available:", torch.cuda.is_available())
print("Tensor test device:", torch.rand(3,3).to(device).device)

# 🔥 Speed boost
torch.backends.cudnn.benchmark = True

# 🔥 Mixed precision
scaler = torch.cuda.amp.GradScaler()


# ── data ─────────────────────────────────────
train_ds = ChestXRayDataset(split="train")
val_ds   = ChestXRayDataset(split="val")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,   # 🔥 try 64 if possible
    shuffle=True,
    num_workers = 0,       # 🔥 increased
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers = 0,
    pin_memory=True,
)

print(f"[train] Train batches : {len(train_loader):,}")
print(f"[train] Val   batches : {len(val_loader):,}")


# ── model ────────────────────────────────────
model = build_model().to(device)
print("Model device:", next(model.parameters()).device)


# ── loss ─────────────────────────────────────
class_weights = compute_class_weights(train_ds.labels).to(device)

if LOSS_TYPE == "focal":
    criterion = FocalLoss(gamma=2.0, alpha=class_weights)
    print("[train] Loss : Focal")
else:
    criterion = nn.BCEWithLogitsLoss(weight=class_weights)
    print("[train] Loss : Weighted BCE")


# ── optimiser ────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


# ── helpers ───────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    loop = tqdm(loader, desc="Train", leave=True)

    for i, (imgs, labels) in enumerate(loop):

        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if i % 100 == 0:
            loop.set_postfix({
                "batch": f"{i}/{len(loader)}",
                "device": str(imgs.device)
            })

        optimizer.zero_grad()

        # 🔥 Mixed precision forward
        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        # 🔥 Scaled backward
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []

    loop = tqdm(loader, desc="Val", leave=True)

    with torch.no_grad():
        for imgs, labels in loop:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = criterion(logits, labels)

            total_loss += loss.item()

            all_preds.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(labels.cpu().numpy())

    preds_np   = np.concatenate(all_preds, axis=0)
    targets_np = np.concatenate(all_targets, axis=0)

    aucs = [
        roc_auc_score(targets_np[:, i], preds_np[:, i])
        for i in range(NUM_LABELS)
        if len(set(targets_np[:, i].tolist())) >= 2
    ]

    return total_loss / len(loader), float(np.mean(aucs)) if aucs else 0.0


# ── training loop ─────────────────────────────
best_auc = 0.0

print("\n" + "=" * 58)
print(f"Training: {NUM_EPOCHS} epochs | batch={BATCH_SIZE} | lr={LR}")
print("=" * 58)

for epoch in range(1, NUM_EPOCHS + 1):

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_auc = validate(model, val_loader, criterion, device)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val   Loss : {val_loss:.4f}")
    print(f"Val   AUC  : {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"→ Saved best model (AUC={best_auc:.4f})")

print("\nTraining complete")
print(f"Best AUC: {best_auc:.4f}")

[train] Device : cuda
[train] GPU    : NVIDIA GeForce RTX 4060 Laptop GPU
[train] VRAM   : 8.6 GB
CUDA available: True
Tensor test device: cuda:0
[dataset] Loaded Data_Entry_2017.csv  →  112,120 rows


C:\Users\adhit\AppData\Local\Temp\ipykernel_85492\650044642.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[dataset] split='train'  →  69,219 images (before subset)
[dataset] Loaded Data_Entry_2017.csv  →  112,120 rows
[dataset] split='val'  →  17,305 images (before subset)
[train] Train batches : 722
[train] Val   batches : 181
[model] 'densenet121'  |  pretrained=True  |  output=14 nodes
[model] Params: 7.0M total  |  7.0M trainable
Model device: cuda:0
[model] Class weights  min=0.84  max=50.00
[train] Loss : Weighted BCE

Training: 10 epochs | batch=96 | lr=0.0001

Epoch 1/10


Train:   0%|          | 0/722 [00:00<?, ?it/s, batch=0/722, device=cuda:0]C:\Users\adhit\AppData\Local\Temp\ipykernel_85492\650044642.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Val:   0%|          | 0/181 [00:00<?, ?it/s]C:\Users\adhit\AppData\Local\Temp\ipykernel_85492\650044642.py:117: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Val: 100%|██████████| 181/181 [00:55<00:00,  3.26it/s]


Train Loss : 2.2744
Val   Loss : 1.4963
Val   AUC  : 0.7940
→ Saved best model (AUC=0.7940)

Epoch 2/10


...

Train Loss : 1.2424
Val   Loss : 1.4561
Val   AUC  : 0.8199

Epoch 9/10


Train:   0%|          | 0/722 [00:00<?, ?it/s, batch=0/722, device=cuda:0]C:\Users\adhit\AppData\Local\Temp\ipykernel_85492\650044642.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Val:   0%|          | 0/181 [00:00<?, ?it/s]C:\Users\adhit\AppData\Local\Temp\ipykernel_85492\650044642.py:117: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Val: 100%|██████████| 181/181 [00:55<00:00,  3.27it/s]


Train Loss : 1.2030
Val   Loss : 1.4516
Val   AUC  : 0.8195

Epoch 10/10


Train:   0%|          | 0/722 [00:00<?, ?it/s, batch=0/722, device=cuda:0]C:\Users\adhit\AppData\Local\Temp\ipykernel_85492\650044642.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Val:   0%|          | 0/181 [00:00<?, ?it/s]C:\Users\adhit\AppData\Local\Temp\ipykernel_85492\650044642.py:117: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Val: 100%|██████████| 181/181 [00:53<00:00,  3.37it/s]


Train Loss : 1.1564
Val   Loss : 1.4979
Val   AUC  : 0.8129

Training complete
Best AUC: 0.8273


---
## 5 · Evaluation
Loads the best checkpoint and runs full evaluation on the **test set**.

In [6]:
"""
evaluate
--------
Outputs:
  - Per-class AUC-ROC table (sorted lowest → highest)
  - Macro-averaged AUC
  - predictions.csv  (raw sigmoid probabilities for every test image)
"""

import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── test data ────────────────────────────────
test_ds     = ChestXRayDataset(split="test")
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         pin_memory=True)

# ── load best model ───────────────────────────
model = build_model().to(device)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
print(f"[evaluate] Loaded: {BEST_MODEL_PATH}")


# ── inference ────────────────────────────────
all_preds, all_targets = [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="  Testing"):
        logits = model(imgs.to(device, non_blocking=True))
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(labels.numpy())

preds_np   = np.concatenate(all_preds,   axis=0)   # [N, 14]
targets_np = np.concatenate(all_targets, axis=0)


# ── per-class AUC table ───────────────────────
print("\n" + "=" * 58)
print("  Per-Class AUC-ROC  (sorted: lowest → highest)")
print("=" * 58)

results = []
for i in range(NUM_LABELS):
    pos_count = int(targets_np[:, i].sum())
    if len(set(targets_np[:, i].tolist())) < 2:
        results.append((LABELS[i], None, pos_count))
        continue
    auc = roc_auc_score(targets_np[:, i], preds_np[:, i])
    results.append((LABELS[i], auc, pos_count))

results.sort(key=lambda x: (x[1] is not None, x[1] if x[1] else 0))

print(f"\n  {'Label':<25} {'AUC':>8}   {'Pos samples':>11}")
print(f"  {'-'*25} {'-'*8}   {'-'*11}")
for label, auc, count in results:
    auc_str = f"{auc:.4f}" if auc is not None else "  N/A  "
    print(f"  {label:<25} {auc_str:>8}   {count:>11,}")

valid_aucs = [a for _, a, _ in results if a is not None]
macro_auc  = float(np.mean(valid_aucs)) if valid_aucs else 0.0

print(f"\n  {'─'*52}")
print(f"  Macro-Averaged AUC : {macro_auc:.4f}   ({len(valid_aucs)}/14 classes)")
print(f"  {'─'*52}")


# ── save predictions CSV ──────────────────────
pred_df = pd.DataFrame(preds_np, columns=LABELS)
pred_df.insert(0, "image_path", test_ds.image_paths)
csv_path = os.path.join(BASE_DIR, "predictions.csv")
pred_df.to_csv(csv_path, index=False)
print(f"\n  Predictions saved → {csv_path}")

[dataset] Loaded Data_Entry_2017.csv  →  112,120 rows
[dataset] split='test'  →  25,596 images (before subset)
[model] 'densenet121'  |  pretrained=True  |  output=14 nodes
[model] Params: 7.0M total  |  7.0M trainable


C:\Users\adhit\AppData\Local\Temp\ipykernel_85492\3553068981.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(BEST_MODEL_PATH, map_locat

[evaluate] Loaded: c:\Users\adhit\Downloads\neurosymbolic\checkpoints\best_model.pth


  Testing:   0%|          | 0/267 [00:00<?, ?it/s]


  Per-Class AUC-ROC  (sorted: lowest → highest)

  Label                          AUC   Pos samples
  ------------------------- --------   -----------
  Pleural Thickening           N/A               0
  Infiltration                0.7097         6,088
  Pneumonia                   0.7108           477
  Consolidation               0.7339         1,815
  No Finding                  0.7340         9,912
  Atelectasis                 0.7526         3,255
  Mass                        0.8114         1,712
  Fibrosis                    0.8187           435
  Effusion                    0.8188         4,648
  Edema                       0.8441           925
  Pneumothorax                0.8614         2,661
  Cardiomegaly                0.8847         1,065
  Emphysema                   0.9120         1,093
  Hernia                      0.9153            86

  ────────────────────────────────────────────────────
  Macro-Averaged AUC : 0.8083   (13/14 classes)
  ────────────────────────────